In [1]:
import torch
from src.qwen3_joint.modeling import C2FForCausalLM
from src.c2f_training.tokenizer import load_or_train_space_tokenizer

/home/ax46/project/coarse-to-fine/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:
mask_type = "autoregressive"

In [15]:
from transformers import PreTrainedTokenizerFast

# --- Load model + tokenizer ---
model = C2FForCausalLM.from_pretrained("checkpoints/decoder")
model.eval()
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
model.to(device)

tokenizer = PreTrainedTokenizerFast.from_pretrained("checkpoints/decoder/tokenizer")
bos_id = tokenizer.bos_token_id or tokenizer.eos_token_id
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id

config = model.config
scale_lengths = config.scale_lengths  # [2, 4, 8, 16, 32]
seq_len = config.seq_len              # 64
content_len = 1 + sum(scale_lengths)  # 63
temperature = 0.8

# --- Generate ---
input_ids = torch.full((1, seq_len), pad_id, dtype=torch.long, device=device)
input_ids[0, 0] = bos_id

if mask_type == "block":
    # Scale-by-scale: each scale sampled independently given coarser scales
    pos = 1
    for i, length in enumerate(scale_lengths):
        with torch.no_grad():
            logits = model(input_ids).logits[0, pos:pos + length, :]
        probs = torch.softmax(logits / temperature, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1).squeeze(-1)
        input_ids[0, pos:pos + length] = sampled
        pos += length

elif mask_type == "causal":
    # Standard autoregressive: left-to-right, one token at a time
    for pos in range(1, content_len):
        with torch.no_grad():
            logits = model(input_ids).logits[0, pos - 1, :]  # shifted: logits[i] predicts token[i+1]
        probs = torch.softmax(logits / temperature, dim=-1)
        sampled = torch.multinomial(probs, num_samples=1)
        input_ids[0, pos] = sampled

# --- Decode + display per-scale ---
pos = 1
names = ["z_4", "z_3", "z_2", "z_1", "text"]
for name, length in zip(names, scale_lengths):
    tokens = input_ids[0, pos:pos + length].tolist()
    words = tokenizer.decode(tokens)
    print(f"{name}: {words}")
    pos += length

z_4: [EOS] [EOS]
z_3: [EOS] [EOS] [EOS] [EOS]
z_2: [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS]
z_1: [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS]
text: [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS] [EOS]
